# Study 2: Statistical Analysis
**IPC vs. Sham — Effects on Exercise Performance and Blood Lactate**

This notebook contains all inferential statistics for Study 2, conducted in R. Analyses include:
- **ANCOVA** for 3600 m and 1200 m time-trial performance
- **ANCOVA** for critical speed and D′
- **Linear mixed model (LME)** for blood lactate across timepoints (log-transformed)
- **Model diagnostics** (residual plots, Cook's distance, DHARMa)

**Data required:** place `Spreadsheet_Study_2.xlsx` inside a `data/` folder in the working directory.

## Setup

In [ ]:
# Install packages only if not already present
pkgs <- c("tidyverse", "readxl", "lme4", "lmerTest",
          "emmeans", "broom", "car", "DHARMa")
install.packages(setdiff(pkgs, rownames(installed.packages())))

library(readxl)
library(tidyverse)
library(lme4)
library(lmerTest)
library(emmeans)
library(broom)
library(car)
library(DHARMa)

## Load and Clean Data

In [ ]:
# Place data file at: data/Spreadsheet_Study_2.xlsx
df <- read_excel("data/Spreadsheet_Study_2.xlsx")

# Set trial factor with Sham as reference level
df <- df %>%
  mutate(trial = factor(trial, levels = c("Sham", "IPC")))

# Clean subject identifiers and remove non-study participant
df <- df %>%
  mutate(
    Subjects = as.character(Subjects),
    Subjects = str_replace_all(Subjects, "\u00A0", " "),  # remove non-breaking spaces
    Subjects = str_squish(Subjects)                        # trim and collapse internal spaces
  ) %>%
  filter(Subjects != "Brian")

df

## ANCOVA — 3600 m Performance
Post-trial 3600 m time (seconds) for IPC vs. Sham, adjusted for baseline performance.

In [ ]:
df2 <- df %>%
  transmute(
    subject   = Subjects,
    group     = factor(trial, levels = c("Sham", "IPC")),
    base_3600 = as.numeric(baseline3600msec),
    post_3600 = as.numeric(`36000msec2`)
  ) %>%
  filter(complete.cases(.))

str(df2)

In [ ]:
m_3600 <- lm(post_3600 ~ group + base_3600, data = df2)

drop1(m_3600, test = "F")

summary(
  contrast(emmeans(m_3600, ~ group), "revpairwise"),
  infer = TRUE
)

## ANCOVA — 1200 m Performance
Post-trial 1200 m time (seconds) for IPC vs. Sham, adjusted for baseline performance.

In [ ]:
df3 <- df %>%
  transmute(
    subject   = Subjects,
    group     = factor(trial, levels = c("Sham", "IPC")),
    base_1200 = as.numeric(baseline1200msec),
    post_1200 = as.numeric(`1200msec2`)
  ) %>%
  filter(complete.cases(.))

str(df3)

In [ ]:
m_1200 <- lm(post_1200 ~ group + base_1200, data = df3)

drop1(m_1200, test = "F")

summary(
  contrast(emmeans(m_1200, ~ group), "revpairwise"),
  infer = TRUE
)

## ANCOVA — Critical Speed and D′
Post-trial critical speed (m/s) and D′ (m) for IPC vs. Sham, each adjusted for its respective baseline.

In [ ]:
df4 <- df %>%
  transmute(
    subject = Subjects,
    group   = factor(trial, levels = c("Sham", "IPC")),
    base_cs = as.numeric(precriticalspeed),
    post_cs = as.numeric(postcriticalspeed),
    base_d  = as.numeric(Pred),
    post_d  = as.numeric(postd)
  ) %>%
  filter(complete.cases(.))

str(df4)

In [ ]:
# Critical speed
m_cs <- lm(post_cs ~ group + base_cs, data = df4)
drop1(m_cs, test = "F")
summary(
  contrast(emmeans(m_cs, ~ group), "revpairwise"),
  infer = TRUE
)

In [ ]:
# D prime
m_d <- lm(post_d ~ group + base_d, data = df4)
drop1(m_d, test = "F")
summary(
  contrast(emmeans(m_d, ~ group), "revpairwise"),
  infer = TRUE
)

## Lactate — Linear Mixed Model
Blood lactate was log-transformed prior to modelling to improve normality of residuals (confirmed by DHARMa diagnostics below). A three-way linear mixed model (group × visit × timepoint) was fit with participant as a random intercept.

Difference-in-differences (DID) contrasts estimate the IPC-specific change from baseline to trial at each timepoint, with Benjamini–Hochberg correction for multiple comparisons.

In [ ]:
lac_cols <- c(
  "T1_Rest_Lac",   "T1_WarmUp_Lac", "T1Post3600Lac", "T1Pre1200Lac", "T1Post1200Lac",
  "T2RestLac",     "T2WarmUpLac",   "T2Post3600Lac", "T2Pre1200Lac", "T2Post1200Lac"
)

lac_long <- df %>%
  transmute(
    subject = Subjects,
    group   = factor(trial, levels = c("Sham", "IPC")),
    across(all_of(lac_cols), as.numeric)
  ) %>%
  pivot_longer(cols = all_of(lac_cols), names_to = "var", values_to = "lactate") %>%
  mutate(
    visit = ifelse(str_starts(var, "T1"), "Baseline", "Post"),
    timepoint = case_when(
      var %in% c("T1_Rest_Lac",   "T2RestLac")    ~ "Rest",
      var %in% c("T1_WarmUp_Lac", "T2WarmUpLac")  ~ "Pre3600",
      var %in% c("T1Post3600Lac", "T2Post3600Lac") ~ "Post3600",
      var %in% c("T1Pre1200Lac",  "T2Pre1200Lac")  ~ "Pre1200",
      var %in% c("T1Post1200Lac", "T2Post1200Lac") ~ "Post1200",
      TRUE ~ NA_character_
    ),
    lac_log   = log(lactate),
    visit     = factor(visit,     levels = c("Baseline", "Post")),
    timepoint = factor(timepoint, levels = c("Rest", "Pre3600", "Post3600", "Pre1200", "Post1200"))
  ) %>%
  filter(!is.na(timepoint), !is.na(lactate))

In [ ]:
m_lac_log <- lmer(
  lac_log ~ group * visit * timepoint + (1 | subject),
  data = lac_long
)

anova(m_lac_log, type = 3)

In [ ]:
emm <- emmeans(m_lac_log, ~ group * visit | timepoint)

# DID = (IPC_post - IPC_baseline) - (Sham_post - Sham_baseline)
# Cell order: Sham Baseline, Sham Post, IPC Baseline, IPC Post
did <- contrast(
  emm,
  method = list(DID = c(1, -1, -1, 1)),
  by = "timepoint"
)

summary(did, infer = TRUE)                 # DID + 95% CI (log scale)
summary(did, infer = TRUE, adjust = "BH")  # BH-adjusted p across 5 timepoints

## Diagnostics — ANCOVA Models
Residual diagnostic plots for the primary ANCOVA model (3600 m). Cook's distance and leverage flag potentially influential observations. The NCV test checks for heteroskedasticity.

In [ ]:
par(mfrow = c(1, 2))
plot(m_3600, which = 1)
qqnorm(resid(m_3600)); qqline(resid(m_3600))
par(mfrow = c(1, 1))

cooks <- cooks.distance(m_3600)
sort(cooks, decreasing = TRUE)
plot(cooks, type = "h"); abline(h = 4 / length(cooks), lty = 2)

hat <- hatvalues(m_3600)
sort(hat, decreasing = TRUE)

car::ncvTest(m_3600)

## Diagnostics — Lactate Model
Simulated residuals via DHARMa confirm the log-transformed mixed model meets distributional assumptions.

In [ ]:
plot(simulateResiduals(m_lac_log))